In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, glob, shutil, hashlib, subprocess, time
from pathlib import Path

DRIVE_ROOT   = Path('/content/drive/MyDrive')
PARENT_DIR   = DRIVE_ROOT / 'CALSHIFT_Research'
PROJECT_ROOT = PARENT_DIR / 'calshift-research'
CRED_DIR     = DRIVE_ROOT / '.gitcreds'

subprocess.run(['git','config','--global','user.name','Md Anas Biswas'], check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'], check=False)
subprocess.run(['git','config','--global','credential.helper','store'], check=False)

for fn, dest in [('.git-credentials','/root/.git-credentials'),
                 ('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR / fn, CRED_DIR / fn):
        if cand.exists():
            shutil.copy(cand, dest); os.chmod(dest, 0o600); break

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
subprocess.run(['git','pull','--ff-only','--quiet'], check=False)

import importlib
if 'config' in sys.modules: importlib.reload(sys.modules['config'])
import config
import numpy as np, pandas as pd
print('ready:', os.getcwd())


Mounted at /content/drive
ready: /content/drive/MyDrive/CALSHIFT_Research/calshift-research


In [2]:
# =============================================================================
# Cell 2 - locate the corrected CIC-IDS2017 (WTMC-2021) files.
# Accepts either CSV or parquet. If nothing is present, prints exactly what to
# download and where to put it, then stops cleanly.
# =============================================================================
CIC17_DIR = config.DATASETS_DIR / 'cicids2017-wtmc2021'
CIC17_DIR.mkdir(parents=True, exist_ok=True)

parquets = sorted(CIC17_DIR.glob('**/*.parquet'))
csvs     = sorted(CIC17_DIR.glob('**/*.csv'))

if not parquets and not csvs:
    print('NO DATA FOUND in', CIC17_DIR)
    print()
    print('Get the WTMC-2021 corrected CIC-IDS2017 (Engelen et al. 2021):')
    print('  Kaggle mirror : https://www.kaggle.com/datasets/dhoogla/distrinetcicids2017')
    print('                  use the WTMC-2021 files (parquet preferred).')
    print('                  do NOT use the later 2022 CNS update files.')
    print('  Original       : https://intrusion-detection.distrinet-research.be/WTMC2021/tools_datasets.html')
    print()
    print('Place the files (any subfolder is fine) under:')
    print(' ', CIC17_DIR)
    print('Then re-run this notebook.')
    raise SystemExit('place the data, then re-run')

USE_PARQUET = bool(parquets)
files = parquets if USE_PARQUET else csvs
print('format :', 'parquet' if USE_PARQUET else 'csv')
print('files  :', len(files))
for p in files:
    print('   ', p.relative_to(CIC17_DIR), f'{p.stat().st_size/1e6:.1f} MB')


format : parquet
files  : 5
    Benign-Monday.parquet 52.4 MB
    Bruteforce-Tuesday.parquet 43.8 MB
    DoS-Wednesday.parquet 66.7 MB
    Infiltration-Webattacks-Thursday.parquet 41.2 MB
    Portscan-DDos-Botnet-Friday.parquet 55.7 MB


In [3]:
# =============================================================================
# Cell 3 - load all day files into one frame.
# Keeps the source file and derives the capture day, which the ladder in a
# later notebook needs (preregistration 10.1: day-aware CIC conditions).
# =============================================================================
DAYS = ['monday','tuesday','wednesday','thursday','friday']

def derive_day(fname):
    low = fname.lower()
    for d in DAYS:
        if d in low:
            return d
    return 'unknown'

frames = []
for p in files:
    d = pd.read_parquet(p) if USE_PARQUET else pd.read_csv(p, low_memory=False)
    d.columns = [c.strip() for c in d.columns]
    d['source_file'] = p.name
    d['day'] = derive_day(p.name)
    frames.append(d)

cic = pd.concat(frames, ignore_index=True)

lab_col = next((c for c in cic.columns if c.lower() == 'label'), None)
assert lab_col is not None, f'no Label column found; columns are {list(cic.columns)[:60]}'
cic = cic.rename(columns={lab_col: 'label_raw'})
cic['label_raw'] = cic['label_raw'].astype(str).str.strip()

print('rows    :', len(cic))
print('columns :', cic.shape[1])
print('days    :', cic['day'].value_counts().to_dict())
if (cic['day'] == 'unknown').any():
    print('WARNING: some files did not encode a weekday in their name:')
    print(cic[cic.day == 'unknown']['source_file'].value_counts().to_dict())


rows    : 1787358
columns : 85
days    : {'wednesday': 477871, 'friday': 370259, 'monday': 350718, 'tuesday': 307071, 'thursday': 281439}


In [5]:
# =============================================================================
# Cell 4 (revised) - label inventory and family mapping (Amendment 5, A5.2),
# adapted to the corrected release's actual label strings.
# Broad family = class; specific attack = subtype.
# =============================================================================
def norm(s):
    s = str(s).lower().strip()
    for ch in ['\u2013', '\u2014', '_']:      # en dash, em dash, underscore
        s = s.replace(ch, '-')
    s = s.replace('-', ' ')
    while '  ' in s:
        s = s.replace('  ', ' ')
    return s.strip()

# normalised specific label -> broad family (the class)
FAMILY_MAP = {
    'benign': 'Benign',
    'attempted relabel as benign': 'Benign',   # payload-absent attempts, relabelled benign by the dataset; retained
    'dos hulk': 'DoS', 'dos goldeneye': 'DoS',
    'dos slowloris': 'DoS', 'dos slowhttptest': 'DoS',
    'ddos': 'DDoS',
    'portscan': 'PortScan', 'port scan': 'PortScan',
    'infiltration portscan': 'PortScan',       # scan phase of Thursday infiltration; grouped by behaviour.
                                               # change to 'Infiltration' to group by campaign instead.
    'ftp patator': 'Brute Force', 'ssh patator': 'Brute Force',
    'web attack brute force': 'Web Attack',
    'web attack xss': 'Web Attack',
    'web attack sql injection': 'Web Attack',
    'botnet': 'Bot', 'bot': 'Bot',
    'infiltration': 'Infiltration', 'infilteration': 'Infiltration',
    'heartbleed': 'Heartbleed',
}

specific = (cic['label_raw']
            .str.replace(r'\s*-\s*Attempted', '', regex=True, case=False)
            .str.strip())
cic['subtype'] = specific
key = specific.map(norm)

unmapped = sorted(set(key[~key.isin(FAMILY_MAP)]))
if unmapped:
    print('UNMAPPED LABELS (add to FAMILY_MAP, then re-run this cell):')
    for u in unmapped:
        print(f'   "{u}"   n={int((key == u).sum())}')
    raise SystemExit('resolve unmapped labels before proceeding')

cic['label'] = key.map(FAMILY_MAP)          # the class

# Only flows that STILL carry an attack label count as attempted. The
# relabel-as-benign flows are benign per the dataset and stay in the primary.
cic['is_attempted'] = (cic['label_raw'].str.contains('attempt', case=False, na=False)
                       & (cic['label'] != 'Benign'))

inv = (cic.groupby(['label', 'subtype', 'is_attempted']).size()
       .rename('n').reset_index().sort_values(['label', 'n'], ascending=[True, False]))
inv.to_csv(config.REPORTS_DIR / 'cicids2017_label_inventory.csv', index=False)

print('families (classes):')
print(cic['label'].value_counts().to_string())
print('\nattempted attack rows (excluded from primary):', int(cic['is_attempted'].sum()))
print('\ninventory ->', config.REPORTS_DIR / 'cicids2017_label_inventory.csv')

families (classes):
label
Benign          1505471
DoS              171755
DDoS              95144
PortScan           7168
Brute Force        6933
Bot                 736
Web Attack          104
Infiltration         36
Heartbleed           11

attempted attack rows (excluded from primary): 0

inventory -> /content/drive/MyDrive/CALSHIFT_Research/calshift-research/reports/cicids2017_label_inventory.csv


In [6]:
# =============================================================================
# Cell 5 - apply the Attempted policy (Amendment 5, A5.1) and save canonical
# frames. Primary EXCLUDES Attempted; a merged copy is saved for the sensitivity
# analysis. Both go to data/interim (gitignored). The class column is "label",
# the subtype column is "subtype".
# =============================================================================
config.INTERIM_DIR.mkdir(parents=True, exist_ok=True)

# primary: drop attempted rows entirely
cic_primary = cic[~cic['is_attempted']].reset_index(drop=True)

# sensitivity: keep attempted, but folded into the parent family (already the
# case, since "label" was mapped from the family regardless of attempted)
cic_merged = cic.reset_index(drop=True)

cic_primary.to_parquet(config.INTERIM_DIR / 'cicids2017_primary.parquet', index=False)
cic_merged.to_parquet(config.INTERIM_DIR / 'cicids2017_attempted_merged.parquet', index=False)

print('PRIMARY (Attempted excluded)')
print('  rows   :', len(cic_primary))
print('  classes:', cic_primary['label'].value_counts().to_dict())
print('\nSENSITIVITY (Attempted merged into parent)')
print('  rows   :', len(cic_merged))
print('\nsaved:')
print('  data/interim/cicids2017_primary.parquet')
print('  data/interim/cicids2017_attempted_merged.parquet')


PRIMARY (Attempted excluded)
  rows   : 1787358
  classes: {'Benign': 1505471, 'DoS': 171755, 'DDoS': 95144, 'PortScan': 7168, 'Brute Force': 6933, 'Bot': 736, 'Web Attack': 104, 'Infiltration': 36, 'Heartbleed': 11}

SENSITIVITY (Attempted merged into parent)
  rows   : 1787358

saved:
  data/interim/cicids2017_primary.parquet
  data/interim/cicids2017_attempted_merged.parquet


In [7]:
# =============================================================================
# Cell 6 - PROJECTED feasibility preview (non-binding).
# The binding feasibility table and focal class are computed per ladder
# condition in the next notebook, because "source" differs by condition
# (Amendment 5, A5.3). This whole-dataset preview only flags which families are
# likely to be excluded by the section 7.6 rule, so there are no surprises.
# =============================================================================
need = config.min_calib_n(config.ALPHA_PRIMARY)
f_src = config.SPLIT_FRACTIONS['source_cal_pool']

attacks = cic_primary[cic_primary['label'] != 'Benign']
proj = (attacks['label'].value_counts() * f_src).round().astype(int).rename('projected_source_calib')
tbl = proj.to_frame()
tbl['min_calib_needed'] = need
tbl['likely_feasible'] = tbl['projected_source_calib'] >= need
print(f'whole-dataset projection at alpha={config.ALPHA_PRIMARY}, need >= {need} per class')
print(tbl.to_string())
print('\nNote: preview only. Binding feasibility is per-condition in notebook 10.')
likely_excluded = tbl[~tbl['likely_feasible']].index.tolist()
print('likely excluded families:', likely_excluded or 'none')


whole-dataset projection at alpha=0.05, need >= 19 per class
              projected_source_calib  min_calib_needed  likely_feasible
label                                                                  
DoS                            25763                19             True
DDoS                           14272                19             True
PortScan                        1075                19             True
Brute Force                     1040                19             True
Bot                              110                19             True
Web Attack                        16                19            False
Infiltration                       5                19            False
Heartbleed                         2                19            False

Note: preview only. Binding feasibility is per-condition in notebook 10.
likely excluded families: ['Web Attack', 'Infiltration', 'Heartbleed']


In [8]:
# =============================================================================
# Cell 7 - record/verify file hashes. Recorded on first run, verified after,
# into the same reports/dataset_hashes.json used by notebook 01.
# =============================================================================
def sha256_of(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()

HASHES_PATH = config.REPORTS_DIR / 'dataset_hashes.json'
HASHES = json.loads(HASHES_PATH.read_text()) if HASHES_PATH.exists() else {}

for p in files:
    k = f'cicids2017/{p.name}'
    h = sha256_of(p)
    if k in HASHES:
        print(('OK      ' if HASHES[k] == h else 'MISMATCH'), k)
        if HASHES[k] != h:
            raise RuntimeError(f'hash changed for {k}')
    else:
        HASHES[k] = h
        print('RECORDED', k)

HASHES_PATH.write_text(json.dumps(HASHES, indent=2))
print('\nhashes ->', HASHES_PATH)


RECORDED cicids2017/Benign-Monday.parquet
RECORDED cicids2017/Bruteforce-Tuesday.parquet
RECORDED cicids2017/DoS-Wednesday.parquet
RECORDED cicids2017/Infiltration-Webattacks-Thursday.parquet
RECORDED cicids2017/Portscan-DDos-Botnet-Friday.parquet

hashes -> /content/drive/MyDrive/CALSHIFT_Research/calshift-research/reports/dataset_hashes.json


In [9]:
# =============================================================================
# Cell 8 - extend the dataset manifest with the CIC-IDS2017 block.
# =============================================================================
man_path = config.REPORTS_DIR / 'dataset_manifest.json'
man = json.loads(man_path.read_text()) if man_path.exists() else {'datasets': {}}
man.setdefault('datasets', {})

man['datasets']['cicids2017'] = {
    'role': 'second environment, corrected CIC-IDS2017 (IEEE CNS 2022 release, Liu et al. 2022; Distrinet, descends from Engelen et al. WTMC-2021)',
    'source_format': 'parquet' if USE_PARQUET else 'csv',
    'n_files': int(len(files)),
    'rows_all': int(len(cic)),
    'rows_primary_attempted_excluded': int(len(cic_primary)),
    'attempted_rows': int(cic['is_attempted'].sum()),
    'attempted_policy': 'exclude (primary); merge (sensitivity)  [Amendment 5 A5.1]',
    'taxonomy': 'family = class, variant = subtype  [Amendment 5 A5.2]',
    'families': sorted(cic_primary['label'].unique().tolist()),
    'days_present': sorted(cic['day'].unique().tolist()),
    'class_counts_primary': {k: int(v) for k, v in cic_primary['label'].value_counts().items()},
}

man_path.write_text(json.dumps(man, indent=2))
print(json.dumps(man['datasets']['cicids2017'], indent=2))


{
  "role": "second environment, corrected CIC-IDS2017 (IEEE CNS 2022 release, Liu et al. 2022; Distrinet, descends from Engelen et al. WTMC-2021)",
  "source_format": "parquet",
  "n_files": 5,
  "rows_all": 1787358,
  "rows_primary_attempted_excluded": 1787358,
  "attempted_rows": 0,
  "attempted_policy": "exclude (primary); merge (sensitivity)  [Amendment 5 A5.1]",
  "taxonomy": "family = class, variant = subtype  [Amendment 5 A5.2]",
  "families": [
    "Benign",
    "Bot",
    "Brute Force",
    "DDoS",
    "DoS",
    "Heartbleed",
    "Infiltration",
    "PortScan",
    "Web Attack"
  ],
  "days_present": [
    "friday",
    "monday",
    "thursday",
    "tuesday",
    "wednesday"
  ],
  "class_counts_primary": {
    "Benign": 1505471,
    "DoS": 171755,
    "DDoS": 95144,
    "PortScan": 7168,
    "Brute Force": 6933,
    "Bot": 736,
    "Web Attack": 104,
    "Infiltration": 36,
    "Heartbleed": 11
  }
}


In [10]:
# =============================================================================
# Cell 9 - commit and push. Data stays in data/ (gitignored); only the
# inventory, hashes and manifest are committed.
# =============================================================================
def git(*args, show=True):
    r = subprocess.run(['git', *args], capture_output=True, text=True)
    if show:
        if r.stdout.strip(): print(r.stdout.strip())
        if r.stderr.strip(): print(r.stderr.strip())
    return r

for s, d in [('/root/.git-credentials', PARENT_DIR / '.git-credentials'),
             ('/root/.gitconfig',       PARENT_DIR / '.gitconfig')]:
    if os.path.exists(s): shutil.copy(s, d)

os.chdir(PROJECT_ROOT)
git('add','-A', show=False)
if git('status','--porcelain', show=False).stdout.strip():
    git('commit','-m','nb09: CIC-IDS2017 acquisition, labels, Amendment 5 decisions')
    r = git('push','-u','origin','main')
    if r.returncode: print('PUSH FAILED. Commit is safe locally.')
else:
    print('nothing to commit')
print(git('log','--oneline','-3', show=False).stdout)


[main d69475b] nb09: CIC-IDS2017 acquisition, labels, Amendment 5 decisions
 6 files changed, 66 insertions(+), 2 deletions(-)
 create mode 100644 kaggle.json
 create mode 100644 notebooks/09_cic_setup.ipynb
 create mode 100644 reports/cicids2017_label_inventory.csv
Branch 'main' set up to track remote branch 'main' from 'origin'.
To https://github.com/anasbiswas1/calshift-research.git
   c21ce51..d69475b  main -> main
d69475b nb09: CIC-IDS2017 acquisition, labels, Amendment 5 decisions
c21ce51 preregistration amendment 5: CIC-IDS2017 second-environment spec
9d12eca amendment 3: disclose placeholder-then-restore sequence



In [11]:
import subprocess, os
from pathlib import Path
os.chdir('/content/drive/MyDrive/CALSHIFT_Research/calshift-research')

def git(*a, show=True):
    r = subprocess.run(['git', *a], capture_output=True, text=True)
    if show and (r.stdout or r.stderr): print((r.stdout + r.stderr).strip())
    return r

print('BEFORE - commits that touch kaggle.json:')
git('log', '--oneline', '--all', '--', 'kaggle.json')

p = Path('kaggle.json')
if p.exists(): p.unlink()
git('rm', '--cached', '--ignore-unmatch', 'kaggle.json', show=False)

gi = Path('.gitignore')
have = gi.read_text() if gi.exists() else ''
if 'kaggle.json' not in have:
    with open('.gitignore', 'a') as f:
        f.write(('' if have == '' or have.endswith('\n') else '\n') + 'kaggle.json\n')
git('add', '.gitignore', show=False)

git('commit', '--amend', '--no-edit')
git('push', '--force-with-lease')

print('\nAFTER - commits that touch kaggle.json (want: nothing):')
git('log', '--oneline', '--all', '--', 'kaggle.json')
print('working tree still has kaggle.json:', Path('kaggle.json').exists())
print('\nrecent commits:')
git('log', '--oneline', '-3')

BEFORE - commits that touch kaggle.json:
d69475b nb09: CIC-IDS2017 acquisition, labels, Amendment 5 decisions
[main d057f5b] nb09: CIC-IDS2017 acquisition, labels, Amendment 5 decisions
 Date: Fri Jul 24 20:09:01 2026 +0000
 6 files changed, 66 insertions(+), 2 deletions(-)
 create mode 100644 notebooks/09_cic_setup.ipynb
 create mode 100644 reports/cicids2017_label_inventory.csv
To https://github.com/anasbiswas1/calshift-research.git
 + d69475b...d057f5b main -> main (forced update)

AFTER - commits that touch kaggle.json (want: nothing):
working tree still has kaggle.json: False

recent commits:
d057f5b nb09: CIC-IDS2017 acquisition, labels, Amendment 5 decisions
c21ce51 preregistration amendment 5: CIC-IDS2017 second-environment spec
9d12eca amendment 3: disclose placeholder-then-restore sequence


CompletedProcess(args=['git', 'log', '--oneline', '-3'], returncode=0, stdout='d057f5b nb09: CIC-IDS2017 acquisition, labels, Amendment 5 decisions\nc21ce51 preregistration amendment 5: CIC-IDS2017 second-environment spec\n9d12eca amendment 3: disclose placeholder-then-restore sequence\n', stderr='')